# Feedback Triage

Fetch thumbs-down feedback from Langfuse and create a human-reviewable triage CSV with Markdown trace files.

## Setup

Required environment variables:

- `LANGFUSE_BASE_URL` defaults to `https://cloud.langfuse.com`
- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_SECRET_KEY`
- `OPENAI_API_KEY` for complaint triage classification

In [1]:
import com.langfuse.client.LangfuseClient
import dev.example.langfuse.FeedbackTriageExporter
import dev.example.langfuse.LangfuseFeedbackClient
import dev.example.langfuse.LlmFeedbackTriager
import dev.dokimos.springai.SpringAiSupport
import org.jetbrains.kotlinx.dataframe.api.*
import org.jetbrains.kotlinx.dataframe.io.readCSV
import org.jetbrains.kotlinx.jupyter.api.HTML
import org.springframework.ai.chat.client.ChatClient
import org.springframework.ai.openai.OpenAiChatModel
import org.springframework.ai.openai.OpenAiChatOptions
import org.springframework.ai.openai.api.OpenAiApi
import java.nio.file.Files
import java.nio.file.Path

In [7]:
val langfuseBaseUrl = System.getenv("LANGFUSE_BASE_URL") ?: "https://cloud.langfuse.com"
val langfusePublicKey = requireNotNull(System.getenv("LANGFUSE_PUBLIC_KEY")) { "LANGFUSE_PUBLIC_KEY must be set" }
val langfuseSecretKey = requireNotNull(System.getenv("LANGFUSE_SECRET_KEY")) { "LANGFUSE_SECRET_KEY must be set" }
val openAiApiKey = requireNotNull(System.getenv("OPENAI_API_KEY")) { "OPENAI_API_KEY must be set" }

val langfuseClient = LangfuseClient.builder()
    .url(langfuseBaseUrl)
    .credentials(langfusePublicKey, langfuseSecretKey)
    .build()

val feedbackClient = LangfuseFeedbackClient(langfuseClient)

val openAiApi = OpenAiApi.builder().apiKey(openAiApiKey).build()
val triageChatModel = OpenAiChatModel.builder()
    .openAiApi(openAiApi)
    .defaultOptions(
        OpenAiChatOptions.builder()
            .model(OpenAiApi.ChatModel.GPT_5_CHAT_LATEST)
            .temperature(0.0)
            .build()
    )
    .build()
val triageJudge = SpringAiSupport.asJudge(ChatClient.builder(triageChatModel))
val feedbackTriager = LlmFeedbackTriager(triageJudge)
val triageExporter = FeedbackTriageExporter(feedbackClient, feedbackTriager)

## Export Negative Feedback

The CSV is the review index. The linked Markdown files contain full trace details.

In [3]:
val export = triageExporter.exportNegativeFeedback(
    limit = 20,
    outputDir = Path.of("eval-data/feedback-triage")
)

println("Exported ${export.exportedRows} rows")
println("CSV: ${export.csvFile.toAbsolutePath()}")
println("Traces: ${export.tracesDirectory.toAbsolutePath()}")

Exported 6 rows
CSV: /Users/urs/development/github/ai/kotlin-edd-talk/spring-ai/src/notebooks/eval-data/feedback-triage/negative-feedback-triage.csv
Traces: /Users/urs/development/github/ai/kotlin-edd-talk/spring-ai/src/notebooks/eval-data/feedback-triage/traces


## Human Review Negative Feedback

Classified feedback records with triage decisions and notes for human reviewers. Use the `traceFile` links to inspect full feedback details when needed.

In [6]:
val triageDf = DataFrame.readCSV(export.csvFile.toFile())
fun recordHtmlEscape(value: String): String = value
    .replace("&", "&amp;")
    .replace("<", "&lt;")
    .replace(">", "&gt;")
    .replace("\"", "&quot;")

fun createTriageHtml(idxOnly:Int? = null): String {
val recordHtml = buildString {
    append("""
        <style>
          .record-list { text-align: left; }
          .record-section { border-top: 4px solid #673ab7; margin: 28px 0 36px; padding-top: 14px; }
          .record-title { margin: 0 0 10px; color: #4527a0; font-size: 18px; font-weight: 700; text-align: left; }
          .record-subtitle { margin: 0 0 12px; color: #555; font-size: 13px; text-align: left; }
          .record-table { border-collapse: collapse; width: 100%; font-size: 14px; text-align: left; }
          .record-table th, .record-table td { border: 1px solid #ddd; padding: 8px; vertical-align: top; }
          .record-table th { background: #f5f5f5; width: 220px; }
          .record-table td, .record-table td * { text-align: left !important; }
          .record-table td { white-space: pre-wrap; direction: ltr; }
        </style>
        <div class=\"record-list\">
    """.trimIndent())

    fun createRow(index:Int) {
        val row = triageDf[index]
        val request = row["request"]?.toString().orEmpty()
        val complaint = row["complaintSummary"]?.toString().orEmpty()
        val title = complaint.ifBlank { request }.take(140)
        append("<section class='record-section'>")
        append("<h3 class='record-title'>Feedback: ${recordHtmlEscape(title)}</h3>")
        append("<p class='record-subtitle'>Trace: ${recordHtmlEscape(row["traceId"]?.toString().orEmpty())}</p>")
        append("<table class='record-table'><tbody>")

        triageDf.columnNames().forEach { column ->
            val rawValue = row[column]?.toString().orEmpty()
            val value = when (column) {
                "traceFile" -> "<a href='${recordHtmlEscape(rawValue)}'>${recordHtmlEscape(rawValue)}</a>"
                "langfuseUrl" -> if (rawValue.isBlank()) "" else "<a href='${recordHtmlEscape(rawValue)}'>Langfuse</a>"
                else -> recordHtmlEscape(rawValue)
            }
            append("<tr><th>${recordHtmlEscape(column)}</th><td>$value</td></tr>")
        }

        append("</tbody></table></section>")
    }
    if(idxOnly!=null) {
        createRow(idxOnly)
    } else
        repeat(triageDf.rowsCount()) { index -> createRow(index) }

    append("</div>")
    }
    return recordHtml
}

HTML(createTriageHtml(2))

traceId,e15c492e9070f51df05cd202857efc04
observationId,e4761ca6315b6fa0
sessionId,77004c07-83d3-4718-a795-69f58301b7fb
timestamp,2026-05-16T15:30:11.375Z
rating,DOWN
request,I'm interested in beginner session about KMP
answer,"Here are some Kotlin Multiplatform (KMP) sessions that might interest you — especially if you’re getting started: 1. **“What Nobody Told Us About Shipping Kotlin to iOS”** 🕑 *May 22, 14:00–14:45* 📍 *Room 13a* 🎤 *Speaker: Suhyeon (Leah) Kim* 👉 A great talk for beginners exploring real-world KMP experiences, especially around iOS integration. 2. **“What’s New in Compose Multiplatform: Better Shared UI for iOS and Beyond”** 🕐 *May 21, 13:00–13:45* 📍 *Keynote, Room 14* 🎤 *Speakers: Sebastian Aigner & Márton Braun* 👉 Perfect if you want to understand how Compose Multiplatform simplifies shared UI development. 3. **“Sony’s KMP Journey: Scaling BLE & Hardware with Kotlin Multiplatform”** 🕐 *May 22, 13:00–13:45* 📍 *Room 13b* 🎤 *Speaker: Sergio Carrilho* 👉 While more intermediate, it’s a great case study to see KMP in action at scale. If you’re **just starting out**, I’d recommend beginning with **“What Nobody Told Us About Shipping Kotlin to iOS”** — it’s practical and beginner-friendly. Would you like me to add that session to your preferences?"
userFeedbackReason,Sessions recommendation of the past
complaintSummary,The user says the assistant recommended sessions that have already occurred.
failureMode,Outdated or stale content presented as current recommendations
affectedCapability,Session recommendation


## Human Review

Open `negative-feedback-triage.csv` and fill these fields manually:

- `humanDecision`: `ok` or `nok`
- `humanNotes`: short reviewer note
- `regressionCandidate`: `yes` for issues that should become evals

Use the `traceFile` column to inspect the full Markdown trace when needed.